In [196]:
import vcfpy
import pandas as pd
import numpy as np

def load_mutect_snv_vcf(vcf_path: str) -> tuple[pd.DataFrame, str, str]:
    """
    Load a Mutect2-style SNV/indel VCF with vcfpy and return:
        df, normal_sample_name, tumor_sample_name

    - One row per ALT allele (multi-allelic sites expanded).
    - For each sample, extracts AD (allelic depths) and DP (depth).
    - Computes normal_vaf and tumor_vaf for that ALT.
    """
    reader = vcfpy.Reader.from_path(vcf_path)
    samples = reader.header.samples.names  # e.g. ["NORMAL", "TUMOR"]
    print("Samples in VCF:", samples)

    # Try to identify normal/tumor explicitly, otherwise fallback to order
    normal_sample = None
    tumor_sample = None
    for s in samples:
        su = s.upper()
        if "NORMAL" in su or "CONTROL" in su:
            normal_sample = s
        if "TUMOR" in su or "TUMOUR" in su:
            tumor_sample = s

    if normal_sample is None and len(samples) >= 1:
        normal_sample = samples[0]
    if tumor_sample is None and len(samples) >= 2:
        tumor_sample = samples[-1]

    print("Assuming normal sample:", normal_sample)
    print("Assuming tumor sample :", tumor_sample)

    rows = []

    for rec in reader:
        chrom = rec.CHROM
        pos   = rec.POS
        vid   = rec.ID[0] if rec.ID else None
        ref   = rec.REF
        qual  = rec.QUAL
        filt  = ";".join(rec.FILTER)

        # Flatten INFO once per record
        info_flat = {}
        for key, value in rec.INFO.items():
            if isinstance(value, list) and len(value) == 1:
                info_flat[key] = value[0]
            else:
                info_flat[key] = value

        # Expand multi-allelic sites: one row per ALT
        for alt_idx, alt_obj in enumerate(rec.ALT):
            alt_allele = getattr(alt_obj, "value", str(alt_obj))

            row = {
                "chrom": chrom,
                "pos": pos,
                "id": vid,
                "ref": ref,
                "alt": alt_allele,
                "alt_index": alt_idx,  # which ALT this row refers to
                "qual": qual,
                "filter": filt,
            }

            # Add flattened INFO
            row.update(info_flat)

            # Per-sample FORMAT fields
            for sample in samples:
                call = rec.call_for_sample[sample]
                data = call.data

                ad = data.get("AD")  # [ref, alt1, alt2, ...]
                dp = data.get("DP")

                if ad is not None and len(ad) > 0:
                    ad_ref = ad[0]
                    ad_alt = ad[alt_idx + 1] if len(ad) > (alt_idx + 1) else np.nan
                else:
                    ad_ref = np.nan
                    ad_alt = np.nan

                row[f"{sample}_AD_ref"] = ad_ref
                row[f"{sample}_AD_alt"] = ad_alt
                row[f"{sample}_DP"]     = dp if dp is not None else np.nan

            rows.append(row)

    df = pd.DataFrame(rows)

    # Ensure numeric columns are numeric where present
    numeric_cols = []
    for sample in samples:
        numeric_cols.extend([
            f"{sample}_AD_ref",
            f"{sample}_AD_alt",
            f"{sample}_DP",
        ])

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Convenience: VAFs for normal + tumor
    if normal_sample is not None:
        ref = df[f"{normal_sample}_AD_ref"]
        alt = df[f"{normal_sample}_AD_alt"]
        df["normal_vaf"] = alt / (ref + alt)
    else:
        df["normal_vaf"] = np.nan

    if tumor_sample is not None:
        ref = df[f"{tumor_sample}_AD_ref"]
        alt = df[f"{tumor_sample}_AD_alt"]
        df["tumor_vaf"] = alt / (ref + alt)
    else:
        df["tumor_vaf"] = np.nan

    df.drop(columns=["ECNT","GERMQ", "MFRL", "NALIGNS", "NALOD", "NLOD", "ROQ", "UNITIGS", "ALIGN_DIFF", "RPA", "RU", "STR", "STRQ"], inplace=True)

    return df, normal_sample, tumor_sample



def high_conf_somatic_mask(
    df: pd.DataFrame,
    min_tumor_vaf: float = 0.05,
    max_normal_vaf: float = 0.02,
    min_tlod: float = 6.0,
    min_popaf: float = 5.0,   # NEW: minimum -log10(AF)
    min_tumor_dp = 20,
    min_normal_dp = 10,
    use_popaf: bool = True,
    require_tlod: bool = False,
) -> pd.Series:
    filt = df["filter"].fillna("")
    pass_mask = (filt == "PASS") | (filt == ".") | (filt == "")

    tumor_vaf = df.get("tumor_vaf", pd.Series(np.nan, index=df.index))
    tumor_vaf_mask = (tumor_vaf >= min_tumor_vaf)

    normal_vaf = df.get("normal_vaf", pd.Series(np.nan, index=df.index))
    normal_vaf_mask = (normal_vaf <= max_normal_vaf) | normal_vaf.isna()

    min_tumor_dp = df.get("TUMOR_DP", pd.Series(np.nan, index=df.index))
    tumor_dp_mask = (min_tumor_dp >= min_tumor_dp) | min_tumor_dp.isna()

    min_normal_dp = df.get("NORMAL_DP", pd.Series(np.nan, index=df.index))
    normal_dp_mask = (min_normal_dp >= min_normal_dp) | min_normal_dp.isna()

    tlod = pd.to_numeric(df.get("TLOD", np.nan), errors="coerce")
    if require_tlod:
        tlod_mask = tlod >= min_tlod
    else:
        tlod_mask = (tlod >= min_tlod) | tlod.isna()

    if use_popaf:
        popaf = pd.to_numeric(df.get("POPAF", np.nan), errors="coerce")
        # KEEP variants that are rare: POPAF high, or unknown
        popaf_mask = (popaf >= min_popaf) | popaf.isna()
    else:
        popaf_mask = pd.Series(True, index=df.index)

    mask = pass_mask & tumor_vaf_mask & normal_vaf_mask & tlod_mask & popaf_mask & tumor_dp_mask & normal_dp_mask
    return mask




vcf_path = r"C:\Users\stavz\Desktop\masters\APM\data\SNV_TCGA\raw_somatic_snv\TCGA-BRCA.ff26324d-a391-43ed-a8d8-5fba1f958da7.wgs.tumor_normal.gatk4_mutect2.raw_somatic_mutation.vcf"
snv_df, normal_sample, tumor_sample = load_mutect_snv_vcf(vcf_path)

mask = high_conf_somatic_mask(
    snv_df,
    min_tumor_vaf=0.05, # tumor_vaf >= 0.05
    max_normal_vaf=0.02, # normal_vaf <= 0.02
    min_tlod=6.0,
    min_popaf=3.0,        # AF ≲ 1e-3
    min_tumor_dp = 20,
    min_normal_dp = 10,
    use_popaf=True,
    require_tlod=False,
)


snv_high_conf = snv_df[mask].copy()




Samples in VCF: ['TUMOR', 'NORMAL']
Assuming normal sample: NORMAL
Assuming tumor sample : TUMOR


In [52]:
snv_high_conf[(snv_high_conf["TUMOR_DP"] >= 20 ) & (snv_high_conf["NORMAL_DP"] >= 10 ) \
& (snv_high_conf["normal_vaf"] <= 0.02 ) & (snv_high_conf["TLOD"] >= 6 ) & (snv_high_conf["POPAF"] >= 3 )  ]

,chrom,pos,id,ref,alt,alt_index,qual,filter,AS_FilterStatus,AS_SB_TABLE,...,TUMOR_AD_alt,TUMOR_DP,NORMAL_AD_ref,NORMAL_AD_alt,NORMAL_DP,RU,STR,STRQ,normal_vaf,tumor_vaf
1,chr1,872566,None,G,A,0,None,PASS,SITE,"60,81|8,4",...,12,120,33,0,33,NaN,NaN,NaN,0.0,0.100000
3,chr1,1746450,None,C,A,0,None,PASS,SITE,"42,42|3,9",...,12,70,26,0,26,NaN,NaN,NaN,0.0,0.171429
4,chr1,2083451,None,C,A,0,None,PASS,SITE,"76,79|8,10",...,18,129,44,0,44,NaN,NaN,NaN,0.0,0.139535
6,chr1,2401230,None,TTCCTC,T,0,None,PASS,SITE,"15,87|1,3",...,4,79,27,0,27,NaN,NaN,NaN,0.0,0.050633
7,chr1,2653735,None,A,G,0,None,PASS,SITE,"148,126|9,3",...,12,221,65,0,65,NaN,NaN,NaN,0.0,0.054299
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4521,chrX,152141495,None,G,A,0,None,PASS,SITE,"79,71|9,5",...,14,119,45,0,45,NaN,NaN,NaN,0.0,0.117647
4522,chrX,152210481,None,G,T,0,None,PASS,SITE,"62,67|12,20",...,32,129,32,0,32,NaN,NaN,NaN,0.0,0.248062
4525,chrX,152657108,None,G,T,0,None,PASS,SITE,"81,82|13,16",...,29,144,48,0,48,NaN,NaN,NaN,0.0,0.201389
4528,chrX,154380614,None,G,A,0,None,PASS,SITE,"91,69|13,7",...,20,145,35,0,35,NaN,NaN,NaN,0.0,0.137931


In [1]:
s, _, _ = load_mutect_snv_vcf(r"data/SNV_TCGA/TCGA-BRCA.0a27e661-7a1c-4172-a521-05a8cc727653.wgs.tumor_normal.gatk4_mutect2.raw_somatic_mutation.APM_1Mb.vep.vcf")

NameError: name 'load_mutect_snv_vcf' is not defined

In [4]:
import io
import vcfpy
import pandas as pd
import numpy as np


def high_conf_somatic_mask(
    df: pd.DataFrame,
    min_tumor_vaf: float = 0.05,
    max_normal_vaf: float = 0.02,
    min_tlod: float = 6.0,
    min_popaf: float = 3.0,   # if POPAF = -log10(AF); keep if >= this
    min_tumor_dp: int = 20,
    min_normal_dp: int = 10,
    use_popaf: bool = True,
    require_tlod: bool = False,
) -> pd.Series:
    
    df_pass = df.get("filter", pd.Series(np.nan, index=df.index))
    pass_mask = df_pass == "PASS"

    # --- VAFs ---
    tumor_vaf = df.get("tumor_vaf", pd.Series(np.nan, index=df.index))
    tumor_vaf_mask = (tumor_vaf >= min_tumor_vaf)

    normal_vaf = df.get("normal_vaf", pd.Series(np.nan, index=df.index))
    normal_vaf_mask = (normal_vaf <= max_normal_vaf) | normal_vaf.isna()

    # --- DEPTH ---
    tumor_dp_series = pd.to_numeric(df.get("TUMOR_DP", np.nan), errors="coerce")
    tumor_dp_mask = (tumor_dp_series >= min_tumor_dp) | tumor_dp_series.isna()

    normal_dp_series = pd.to_numeric(df.get("NORMAL_DP", np.nan), errors="coerce")
    normal_dp_mask = (normal_dp_series >= min_normal_dp) | normal_dp_series.isna()

    # --- TLOD ---
    tlod = pd.to_numeric(df.get("TLOD", np.nan), errors="coerce")
    if require_tlod:
        tlod_mask = tlod >= min_tlod
    else:
        tlod_mask = (tlod >= min_tlod) | tlod.isna()

    # --- POPAF ---
    if use_popaf:
        popaf = pd.to_numeric(df.get("POPAF", np.nan), errors="coerce")
        popaf_mask = (popaf >= min_popaf) | popaf.isna()
     
    else:
        popaf_mask = pd.Series(True, index=df.index)

    mask = (
        pass_mask
        & tumor_vaf_mask
        & normal_vaf_mask
        & tumor_dp_mask
        & normal_dp_mask
        & tlod_mask
        & popaf_mask
    )

    return mask


def add_vep_hits_columns(df: pd.DataFrame, csq_description: str, primary_genes: list[str]) -> pd.DataFrame:
    """
    From a df that contains a 'CSQ' column and one row per (variant, ALT),
    add the following columns:

        - gene_hits       : list[dict]  (for Feature_type == "Transcript")
        - regulatory_hits : list[dict]  (Feature_type == "RegulatoryFeature")
        - motif_hits      : list[dict]  (Feature_type == "MotifFeature")

    Summary columns:

        - gene_symbols               : list of unique gene SYMBOLs hit by this variant/ALT
        - hits_canonical             : bool, hits any transcript with CANONICAL == "1"

        Global impact flags (any transcript):
        - has_missense
        - has_nonsense
        - has_frameshift
        - has_splice_effect

        Canonical-specific impact flags (CANONICAL == "1"):
        - has_missense_canonical
        - has_nonsense_canonical
        - has_frameshift_canonical
        - has_splice_effect_canonical

        MANE_SELECT-specific impact flags (MANE_SELECT not None):
        - has_missense_mane
        - has_nonsense_mane
        - has_frameshift_mane
        - has_splice_effect_mane
    """

    # 1) Parse CSQ column names from description
    if "Format:" in csq_description:
        fmt = csq_description.split("Format:")[1].strip().strip('"').strip()
    else:
        fmt = csq_description.strip().strip('"').strip()

    csq_cols = fmt.split("|")
    csq_index = {name: i for i, name in enumerate(csq_cols)}

    def get_field(parts, name, default=None):
        """
        Safely pull a field by VEP name from the parts list.
        Handles missing columns or truncated entries gracefully.
        """
        idx = csq_index.get(name)
        if idx is None:
            return default
        if idx >= len(parts):
            return default
        val = parts[idx]
        if val == "":
            return default
        return val

    # Fields for inner dicts

    GENE_FIELDS = [
        "Allele",
        "Consequence",
        "IMPACT",
        "SYMBOL",
        "Gene",
        "Feature_type",
        "Feature",      # transcript ID
        "BIOTYPE",
        "EXON",
        "INTRON",
        "HGVSc",
        "HGVSp",
        "cDNA_position",
        "CDS_position",
        "Protein_position",
        "Amino_acids",
        "Codons",
        "Existing_variation",
        "VARIANT_CLASS",
        "CANONICAL",
        "MANE",
        "MANE_SELECT",
        "MANE_PLUS_CLINICAL",
        "TSL",
        "APPRIS",
        "CCDS",
        "ENSP",
        "SWISSPROT",
        "TREMBL",
        "UNIPARC",
        "UNIPROT_ISOFORM",
        "GENE_PHENO",
        "SIFT",
        "PolyPhen",
        "DOMAINS",
        "AF",
        "gnomADe_AF",
        "gnomADg_AF",
        "MAX_AF",
        "CLIN_SIG",
        "SOMATIC",
        "PHENO",
        "PUBMED",
    ]

    REGULATORY_FIELDS = [
        "Allele",
        "Consequence",
        "IMPACT",
        "SYMBOL",
        "Gene",
        "Feature_type",
        "Feature",          # ENSR...
        "BIOTYPE",
        "Existing_variation",
        "DISTANCE",
        "VARIANT_CLASS",
        "AF",
        "gnomADe_AF",
        "gnomADg_AF",
        "MAX_AF",
        "CLIN_SIG",
        "SOMATIC",
        "PHENO",
        "PUBMED",
    ]

    MOTIF_FIELDS = [
        "Allele",
        "Consequence",
        "IMPACT",
        "SYMBOL",
        "Gene",
        "Feature_type",
        "Feature",
        "Existing_variation",
        "VARIANT_CLASS",
        "MOTIF_NAME",
        "MOTIF_POS",
        "HIGH_INF_POS",
        "MOTIF_SCORE_CHANGE",
        "TRANSCRIPTION_FACTORS",
        "AF",
        "gnomADe_AF",
        "gnomADg_AF",
        "MAX_AF",
    ]

    def has_consequence(gene_hits, target_terms:
                        set[str]) -> bool:
        """
        Check if ANY transcript (anywhere in gene_hits) has a consequence in target_terms.
        VEP uses '&' to separate multiple SO terms in one 'Consequence' field.
        """
        for hit in gene_hits:
            cons = hit.get("Consequence") or ""
            for term in cons.split("&"):
                if term.strip() in target_terms:
                    return True
        return False

    def has_consequence_filtered(
        gene_hits,
        target_terms: set[str],
        filter_field: str,
        filter_predicate
    ) -> bool:
        """
        Check if ANY transcript that satisfies filter_predicate(hit[filter_field])
        has a consequence in target_terms.
        e.g. filter_field = "CANONICAL", filter_predicate = lambda v: v == "1"
             filter_field = "MANE_SELECT", filter_predicate = lambda v: v is not None
        """
        for hit in gene_hits:
            if not filter_predicate(hit.get(filter_field)):
                continue
            cons = hit.get("Consequence") or ""
            for term in cons.split("&"):
                if term.strip() in target_terms:
                    return True
        return False

    def parse_csq_for_row(row: pd.Series, primary_genes) -> pd.Series:
        """
        For a single row (variant+ALT), build:
          - gene_hits, regulatory_hits, motif_hits
          - gene_symbols, hits_canonical
          - global and canonical/MANE impact flags
        """
        alt_allele = str(row["alt"])

        val = row.get("CSQ")
        if val is None or (isinstance(val, float) and np.isnan(val)):
            return pd.Series(
                {
                    "gene_hits": [],
                    "regulatory_hits": [],
                    "motif_hits": [],
                    "gene_symbols": [],
                    "hits_canonical": False,
                    "has_missense": False,
                    "has_nonsense": False,
                    "has_frameshift": False,
                    "has_splice_effect": False,
                    "has_missense_canonical": False,
                    "has_nonsense_canonical": False,
                    "has_frameshift_canonical": False,
                    "has_splice_effect_canonical": False,
                    "has_missense_mane": False,
                    "has_nonsense_mane": False,
                    "has_frameshift_mane": False,
                    "has_splice_effect_mane": False,
                }
            )

        # CSQ might be a list or a comma-separated string
        if isinstance(val, list):
            entries = val
        else:
            entries = str(val).split(",")

        gene_hits = []
        regulatory_hits = []
        motif_hits = []

        for entry in entries:
            entry = entry.strip()
            if not entry:
                continue

            parts = entry.split("|")

            allele = get_field(parts, "Allele")
            if allele is None:
                continue

            # keep only lines corresponding to this ALT
            if allele != alt_allele:
                continue

            feature_type = get_field(parts, "Feature_type")
            if feature_type is None:
                continue

            if feature_type == "Transcript":
                hit = {}
                gene__name = get_field(parts, "SYMBOL")
                if gene__name not in primary_genes:
                    continue
                for fname in GENE_FIELDS:
                    hit[fname] = get_field(parts, fname)
                gene_hits.append(hit)

            elif feature_type == "RegulatoryFeature":
                hit = {}
                for fname in REGULATORY_FIELDS:
                    hit[fname] = get_field(parts, fname)
                regulatory_hits.append(hit)

            elif feature_type == "MotifFeature":
                hit = {}
                for fname in MOTIF_FIELDS:
                    hit[fname] = get_field(parts, fname)
                motif_hits.append(hit)

        # outer gene list: unique SYMBOLs from gene_hits
        gene_symbol_list  = sorted(
            {
                h.get("SYMBOL")
                for h in gene_hits
                if h.get("SYMBOL") is not None
            }
        )
        gene_symbols = ",".join(gene_symbol_list) if gene_symbol_list else ""


        # canonical flag: any transcript with CANONICAL == "1"
        hits_canonical = any(
            (h.get("CANONICAL") == "1")
            for h in gene_hits
        )

        # --- global consequence flags (any transcript) ---
        MISSENSE = {"missense_variant"}
        NONSENSE = {"stop_gained"}
        FRAMESHIFT = {"frameshift_variant"}
        SPLICE = {
            "splice_acceptor_variant",
            "splice_donor_variant",
            "splice_region_variant",
        }

        has_missense = has_consequence(gene_hits, MISSENSE)
        has_nonsense = has_consequence(gene_hits, NONSENSE)
        has_frameshift = has_consequence(gene_hits, FRAMESHIFT)
        has_splice = has_consequence(gene_hits, SPLICE)

        # --- canonical-specific flags (CANONICAL == "1") ---
        canonical_pred = lambda v: v == "1"

        has_missense_canonical = has_consequence_filtered(
            gene_hits, MISSENSE, "CANONICAL", canonical_pred
        )
        has_nonsense_canonical = has_consequence_filtered(
            gene_hits, NONSENSE, "CANONICAL", canonical_pred
        )
        has_frameshift_canonical = has_consequence_filtered(
            gene_hits, FRAMESHIFT, "CANONICAL", canonical_pred
        )
        has_splice_canonical = has_consequence_filtered(
            gene_hits, SPLICE, "CANONICAL", canonical_pred
        )

        # --- MANE_SELECT-specific flags (MANE_SELECT not None) ---
        mane_pred = lambda v: v is not None

        has_missense_mane = has_consequence_filtered(
            gene_hits, MISSENSE, "MANE_SELECT", mane_pred
        )
        has_nonsense_mane = has_consequence_filtered(
            gene_hits, NONSENSE, "MANE_SELECT", mane_pred
        )
        has_frameshift_mane = has_consequence_filtered(
            gene_hits, FRAMESHIFT, "MANE_SELECT", mane_pred
        )
        has_splice_mane = has_consequence_filtered(
            gene_hits, SPLICE, "MANE_SELECT", mane_pred
        )

        return pd.Series(
            {
                "gene_hits": gene_hits,
                "regulatory_hits": regulatory_hits,
                "motif_hits": motif_hits,
                "gene_symbols": gene_symbols,
                "hits_canonical": hits_canonical,
                "has_missense": has_missense,
                "has_nonsense": has_nonsense,
                "has_frameshift": has_frameshift,
                "has_splice_effect": has_splice,
                "has_missense_canonical": has_missense_canonical,
                "has_nonsense_canonical": has_nonsense_canonical,
                "has_frameshift_canonical": has_frameshift_canonical,
                "has_splice_effect_canonical": has_splice_canonical,
                "has_missense_mane": has_missense_mane,
                "has_nonsense_mane": has_nonsense_mane,
                "has_frameshift_mane": has_frameshift_mane,
                "has_splice_effect_mane": has_splice_mane,
            }
        )

    hits_df = df.apply(parse_csq_for_row, axis=1, primary_genes=primary_genes)
    df = pd.concat([df, hits_df], axis=1)
    return df


def match_cCREs(snv_df: pd.DataFrame, elements: pd.DataFrame) -> pd.DataFrame: 
    df = snv_df.copy()
    elem_hits_col = []
    for idx, sv in snv_df.iterrows():
        chrom = sv["chrom"]
        snv_pos = int(sv["pos"])
        # candidate elements: same chromosome, roughly within the window
        e_chr = elements[elements["chrom"] == chrom].copy()
        e_chr = e_chr[
            (e_chr["end"]   >= snv_pos) &
            (e_chr["start"] <= snv_pos)
        ]

        if e_chr.empty:
            elem_hits_col.append([])  # no elements nearby
            continue

        connections = []

        for _, e in e_chr.iterrows():
            elem_start = int(e["start"])
            elem_end   = int(e["end"])
            cCRE_id    = e["cCRE_id"]
            elem_type  = e.get("raw_type", None)
            genes_by_exact_distance = e.get("genes_by_exact_distance", None)
          
          

            elem_hit = {
                "cCRE_id": cCRE_id,
                "elem_type": elem_type,
                "chrom": chrom,
                "elem_start": elem_start,
                "elem_end": elem_end,
                "genes_by_exact_distance": genes_by_exact_distance,
            }

            connections.append(elem_hit)

        elem_hits_col.append(connections)

    # attach the list-of-dicts back into the DEL subset
    snv_df["cCRE_hits"] = elem_hits_col

    # and write that column back into the full strict_sv_set
    df.loc[snv_df.index, "cCRE_hits"] = snv_df["cCRE_hits"]

    return df



def load_mutect_snv_vcf(
    vcf_path: str, primary_genes: list[str], elements_path: str,
    clean_header: bool = True) -> tuple[pd.DataFrame, str, str]:
    """
    Load a Mutect2-style SNV/indel VCF (optionally VEP-annotated) and return:
        df, normal_sample_name, tumor_sample_name

    - One row per ALT allele (multi-allelic sites expanded).
    - For each sample, extracts AD (allelic depths) and DP (depth).
    - Computes normal_vaf and tumor_vaf for that ALT.
    """
    elements = pd.read_csv(elements_path)

    if clean_header:
        cleaned_lines = []
        with open(vcf_path, "r") as f:
            # 1) Keep only proper "##key=value" meta lines
            for line in f:
                if line.startswith("##"):
                    # Only keep if it contains '='  (valid VCF meta line)
                    if "=" in line:
                        cleaned_lines.append(line)
                    # otherwise skip (e.g. "## ENSEMBL VARIANT EFFECT PREDICTOR v115.2",
                    # "## Output produced at ...")
                    continue

                # 2) First non-## line we care about is "#CHROM ..."
                if "#CHROM" in line.strip():
                    cleaned_lines.append(line)
                    break

                # Any other lines starting with "#" before #CHROM are nonstandard comments -> skip
                # e.g. "# Uploaded variation file", "# Extra comment", etc.
                # So we just continue the loop without appending them.

            # 3) Copy the rest of the file (variant records)
            for line in f:
                cleaned_lines.append(line)

        buffer = io.StringIO("".join(cleaned_lines))
        reader = vcfpy.Reader.from_stream(buffer)
    else:
        reader = vcfpy.Reader.from_path(vcf_path)

    samples = reader.header.samples.names  # e.g. ["NORMAL", "TUMOR"]
    print("Samples in VCF:", samples)


    

    # Try to identify normal/tumor explicitly, otherwise fallback to order
    normal_sample = None
    tumor_sample = None
    for sname in samples:
        su = sname.upper()
        if "NORMAL" in su or "CONTROL" in su:
            normal_sample = sname
        if "TUMOR" in su or "TUMOUR" in su:
            tumor_sample = sname

    if normal_sample is None and len(samples) >= 1:
        normal_sample = samples[0]
    if tumor_sample is None and len(samples) >= 2:
        tumor_sample = samples[-1]

    print("Assuming normal sample:", normal_sample)
    print("Assuming tumor sample :", tumor_sample)

    rows = []

    for rec in reader:
        chrom = rec.CHROM
        pos   = rec.POS
        vid   = rec.ID[0] if rec.ID else None
        ref   = rec.REF
        qual  = rec.QUAL
        filt  = ";".join(rec.FILTER)

        # Flatten INFO once per record
        info_flat = {}
        for key, value in rec.INFO.items():
            if isinstance(value, list) and len(value) == 1:
                info_flat[key] = value[0]
            else:
                info_flat[key] = value

        # Expand multi-allelic sites: one row per ALT
        for alt_idx, alt_obj in enumerate(rec.ALT):
            alt_allele = getattr(alt_obj, "value", str(alt_obj))

            row = {
                "chrom": chrom,
                "pos": pos,
                "id": vid,
                "ref": ref,
                "alt": alt_allele,
                "alt_index": alt_idx,
                "qual": qual,
                "filter": filt,
            }

            # Add flattened INFO
            row.update(info_flat)

            # Per-sample FORMAT fields
            for sample in samples:
                call = rec.call_for_sample[sample]
                data = call.data

                ad = data.get("AD")  # [ref, alt1, alt2, ...]
                dp = data.get("DP")

                if ad is not None and len(ad) > 0:
                    ad_ref = ad[0]
                    ad_alt = ad[alt_idx + 1] if len(ad) > (alt_idx + 1) else np.nan
                else:
                    ad_ref = np.nan
                    ad_alt = np.nan

                row[f"{sample}_AD_ref"] = ad_ref
                row[f"{sample}_AD_alt"] = ad_alt
                row[f"{sample}_DP"]     = dp if dp is not None else np.nan

            rows.append(row)

    df = pd.DataFrame(rows)

    # Ensure numeric columns are numeric where present
    numeric_cols = []
    for sample in samples:
        numeric_cols.extend([
            f"{sample}_AD_ref",
            f"{sample}_AD_alt",
            f"{sample}_DP",
        ])

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Convenience: VAFs for normal + tumor
    if normal_sample is not None:
        ref = df[f"{normal_sample}_AD_ref"]
        alt = df[f"{normal_sample}_AD_alt"]
        df["normal_vaf"] = alt / (ref + alt)
    else:
        df["normal_vaf"] = np.nan

    if tumor_sample is not None:
        ref = df[f"{tumor_sample}_AD_ref"]
        alt = df[f"{tumor_sample}_AD_alt"]
        df["tumor_vaf"] = alt / (ref + alt)
    else:
        df["tumor_vaf"] = np.nan
    
    csq_descr = (
    'Consequence annotations from Ensembl VEP. Format: '
    'Allele|Consequence|IMPACT|SYMBOL|Gene|Feature_type|Feature|BIOTYPE|EXON|INTRON|HGVSc|HGVSp|cDNA_position|CDS_position|Protein_position|Amino_acids|Codons|Existing_variation|DISTANCE|STRAND|FLAGS|VARIANT_CLASS|SYMBOL_SOURCE|HGNC_ID|CANONICAL|MANE|MANE_SELECT|MANE_PLUS_CLINICAL|TSL|APPRIS|CCDS|ENSP|SWISSPROT|TREMBL|UNIPARC|ARC|UNIPROT_ISOFORM|GENE_PHENO|SIFT|PolyPhen|DOMAINS|miRNA|HGVS_OFFSET|AF|AFR_AF|AMR_AF|EAS_AF|EUR_AF|SAS_AF|gnomADe_AF|gnomADe_AFR_AF|gnomADe_AMR_AF|gnomADe_ASJ_AF|gnomADe_EAS_AF|gnomADe_FIN_AF|gnomADe_MID_AF|gnomADe_NFE_AF|gnomADe_REMAINING_AF|gnomADe_SAS_AF|gnomADg_AF|gnomADg_AFR_AF|gnomADg_AMI_AF|gnomADg_AMR_AF|gnomADg_ASJ_AF|gnomADg_EAS_AF|gnomADg_FIN_AF|gnomADg_MID_AF|gnomADg_NFE_AF|gnomADg_REMAINING_AF|gnomADg_SAS_AF|MAX_AF|MAX_AF_POPS|CLIN_SIG|SOMATIC|PHENO|PUBMED|MOTIF_NAME|MOTIF_POS|HIGH_INF_POS|MOTIF_SCORE_CHANGE|TRANSCRIPTION_FACTORS'
    )
    df = add_vep_hits_columns(df, csq_descr, primary_genes)
    df.drop_duplicates(subset=["chrom","pos","ref","alt"], inplace=True)

    mask = high_conf_somatic_mask(
    df,
    min_tumor_vaf=0.05, # tumor_vaf >= 0.05
    max_normal_vaf=0.02, # normal_vaf <= 0.02
    min_tlod=6.0,
    min_popaf=3.0,        # AF ≲ 1e-3
    min_tumor_dp = 20,
    min_normal_dp = 10,
    use_popaf=True,
    require_tlod=False,
    )
    df = df[mask]
    df = df[df["filter"] == "PASS"]

    df = match_cCREs(df, elements)

    return df, normal_sample, tumor_sample


In [5]:

vep_path = "/home/stavz/masters/gdc/SNV/vep_vcfs/TCGA-BRCA.0a27e661-7a1c-4172-a521-05a8cc727653.wgs.tumor_normal.gatk4_mutect2.raw_somatic_mutation.APM_1Mb.vep.vcf"
elements_path = "/home/stavz/masters/gdc/APM/test1/regulatory_elements_matching/regulatory_element_focus_with_evidence.csv"
primary_genes = pd.read_csv("data/primary_genes_only.csv")["gene_name"].tolist()
s, _, _ = load_mutect_snv_vcf(vep_path, primary_genes=primary_genes, elements_path=elements_path)
print("CSQ in columns?", "CSQ" in s.columns)


Samples in VCF: ['TUMOR', 'NORMAL']
Assuming normal sample: NORMAL
Assuming tumor sample : TUMOR
CSQ in columns? True


In [14]:
s["gene_hits"].value_counts()

gene_hits
[]                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            